# Getting Started with Qiskit

**Qiskit Fall Fest 2026 @ UT San Antonio · Day 1 · Thursday, Nov 19, 2026**

Companion notebook for the session *Getting Started with Qiskit*.

| Part | Topic | Qiskit pattern step |
|---|---|---|
| 1 | Set up and check your version | |
| 2 | Building circuits | **Map** |
| 3 | Looking inside: `quantum_info`, qsphere, Bloch sphere | *(check the math)* |
| 4 | Primitives: Sampler and Estimator | **Execute** and **post-process** |
| 5 | Transpiling and noise | **Optimize** |

**How to use it:** run the cells from top to bottom (`Shift + Enter`). Exercises are marked ✏️; the `check()` helper tells you when your circuit is right, and full solutions are at the very end. Try first!


## Part 1 · Set up

In [ ]:
%pip install --upgrade 'qiskit[visualization]' qiskit-aer qiskit-ibm-runtime -q

In [ ]:
from platform import python_version
import qiskit, qiskit_aer, qiskit_ibm_runtime

print(f"Python:             {python_version()}")
print(f"Qiskit:             {qiskit.__version__}   (must start with 2.)")
print(f"qiskit-aer:         {qiskit_aer.__version__}")
print(f"qiskit-ibm-runtime: {qiskit_ibm_runtime.__version__}")

The imports used in this notebook:

In [ ]:
from math import pi
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import Statevector, Operator, SparsePauliOp
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit.visualization import (plot_histogram, plot_distribution, plot_bloch_multivector,
                                  plot_state_qsphere, array_to_latex)

A small helper for the exercises: it compares the state your circuit prepares with a target state, ignoring an overall global phase.

In [ ]:
def check(qc, target, name="state"):
    ok = Statevector(qc).equiv(Statevector(target))
    print(f"{name}: {'correct! 🎉' if ok else 'not yet, keep trying'}")

---
## Part 2 · Map: building circuits

### 2.1 Your first circuit

Every qubit starts in $\lvert 0\rangle$. Gates are **methods** of the circuit; the argument is the **qubit index**. Each call **appends** a gate, so the order of the lines is the order in time.

This is the identity from *Understanding Quantum Circuits*: $HZH = X$.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.z(0)
qc.h(0)

print(qc)          # text drawing

In [ ]:
qc.draw("mpl")     # matplotlib drawing

### 2.2 Qubits, classical bits and registers

`QuantumCircuit(n, m)` makes `n` qubits (register `q`) and `m` classical bits (register `c`). Indices start at 0.

In [ ]:
qc2 = QuantumCircuit(2, 2)
print(qc2.num_qubits, qc2.num_clbits)
print(qc2.qubits)

### 2.3 The gate menu

| Method | Gate | Matrix or action |
|---|---|---|
| `x(q)`, `y(q)`, `z(q)` | Paulis | $X=\begin{pmatrix}0&1\\1&0\end{pmatrix}$, $Y=\begin{pmatrix}0&-i\\i&0\end{pmatrix}$, $Z=\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ |
| `h(q)` | Hadamard | $\tfrac{1}{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ |
| `s(q)`, `t(q)`, `sdg(q)` | phase gates | $\mathrm{diag}(1, i)$, $\mathrm{diag}(1, e^{i\pi/4})$, $S^\dagger$ |
| `rx(θ, q)`, `ry(θ, q)`, `rz(θ, q)` | rotations | $R_y(\theta)=\begin{pmatrix}\cos\frac\theta2&-\sin\frac\theta2\\\sin\frac\theta2&\cos\frac\theta2\end{pmatrix}$ |
| `cx(c, t)`, `cz(c, t)` | CNOT, CZ | flip / phase the target when the control is 1 |
| `swap(a, b)`, `ccx(c1, c2, t)` | SWAP, Toffoli | exchange two qubits; flip `t` when both controls are 1 |

**Angle first, qubit last:** `ry(θ, 0)`. **Control first, target second:** `cx(0, 1)`.

### 2.4 Measuring: `measure_all()` vs `measure()`

The **name** of the classical register matters: you use it to read results in Part 4.

In [ ]:
a = QuantumCircuit(2)
a.h(0)
a.measure_all()            # adds a barrier and a NEW register named 'meas'
print(a.cregs)
a.draw("mpl")

In [ ]:
b = QuantumCircuit(2, 2)
b.h(0)
b.measure([0, 1], [0, 1])  # writes into YOUR register, named 'c'
print(b.cregs)
b.draw("mpl")

### 2.5 Loops build big circuits: the GHZ state

It's just Python. This is our running example:

$$
\lvert000\rangle \xrightarrow{H_0} \tfrac{1}{\sqrt2}(\lvert000\rangle + \lvert001\rangle) \xrightarrow{\mathrm{CX}_{0\to1},\ \mathrm{CX}_{1\to2}} \tfrac{1}{\sqrt2}(\lvert000\rangle + \lvert111\rangle)
$$

In [ ]:
n = 3
ghz = QuantumCircuit(n)
ghz.h(0)
for i in range(n - 1):
    ghz.cx(i, i + 1)

ghz.draw("mpl")

### 2.6 Inspecting a circuit

* `size()`: total number of gates
* `depth()`: number of **layers** (gates on different qubits can share a layer)
* `count_ops()`: gates by type
* `num_nonlocal_gates()`: two-qubit gates, the noisiest kind on real hardware

In [ ]:
print(ghz.depth())
print(ghz.size())
print(ghz.count_ops())
print(ghz.num_nonlocal_gates())

**Poll from the slides:** predict `p.depth()` before running the cell.

In [ ]:
p = QuantumCircuit(2)
p.h(0)
p.h(1)
p.cx(0, 1)
p.x(0)
print("depth:", p.depth(), "  size:", p.size())
p.draw("mpl")

### 2.7 Composing circuits: undo with `inverse()`

`inverse()` reverses the gate order **and** daggers each gate, because $(ABC)^\dagger = C^\dagger B^\dagger A^\dagger$. So $U^\dagger U = I$.

In [ ]:
u = QuantumCircuit(2)
u.h(0)
u.cx(0, 1)
u.t(1)
round_trip = u.compose(u.inverse())

print(Operator(round_trip).equiv(Operator(QuantumCircuit(2))))   # True: it's the identity
round_trip.draw("mpl")

### 2.8 Parameters: one circuit, many angles

A `Parameter` is a placeholder. Build once, choose the value later.

In [ ]:
theta = Parameter("θ")
dial = QuantumCircuit(1)
dial.ry(theta, 0)
dial.measure_all()

print(dial.parameters)
bound = dial.assign_parameters({theta: pi / 2})   # returns a NEW circuit
bound.draw("mpl")

### ✏️ Try it (Part 2)

1. Build a **4-qubit** GHZ circuit with the loop above. What are its `depth()` and `size()`?
2. Build a 3-qubit circuit with a Hadamard on every qubit. Why is its depth **1**?

In [ ]:
# your code here

---
## Part 3 · Look inside: `quantum_info`

### 3.1 The state vector

`Statevector` stores all $2^n$ amplitudes exactly. It needs a circuit **without** measurements.

> NumPy 2 prints `np.str_('000')` and `np.float64(0.5)`; these are just `'000'` and `0.5`.

In [ ]:
sv = Statevector.from_instruction(ghz)
print(sv)
print(sv.probabilities_dict())

Prettier views of the same state:

In [ ]:
display(sv.draw("latex"))     # as a ket
display(array_to_latex(sv))   # as a column vector

### 3.2 Single-qubit states: probability $p$ and phase $\phi$

The most general single-qubit state (up to a global phase) is

$$
\lvert\psi\rangle = \sqrt{1-p}\,\lvert0\rangle + e^{i\phi}\sqrt{p}\,\lvert1\rangle .
$$

Measuring gives 1 with probability $p$. The **relative phase** $\phi$ doesn't change the probabilities, but it is what makes interference possible.

The **qsphere** draws each basis state as a dot: the dot's **size** is its probability and its **colour** is its phase. Below: $\lvert0\rangle$, $\lvert1\rangle$, $\lvert+\rangle$ and $\tfrac{1}{\sqrt2}(\lvert0\rangle + i\lvert1\rangle)$ (same $p = \tfrac12$, but $\phi = 90^\circ$).

In [ ]:
from numpy import sqrt
display(plot_state_qsphere(Statevector([1, 0])))
display(plot_state_qsphere(Statevector([0, 1])))
display(plot_state_qsphere(Statevector([1 / sqrt(2), 1 / sqrt(2)])))
display(plot_state_qsphere(Statevector([1 / sqrt(2), 1j / sqrt(2)]),
                           show_state_phases=True, use_degrees=True))

The phase gates move a state **around** the equator: starting from $\lvert+\rangle$, $S$ adds $90^\circ$ to $\phi$, $Z$ adds $180^\circ$, and $S^\dagger$ (`sdg`) subtracts $90^\circ$. To build intuition, drop gates on a qubit in the [IBM Quantum Composer](https://quantum.cloud.ibm.com/composer) and watch the state change.

#### ✏️ Practice: reach these states, starting from $\lvert 0\rangle$

i) $\lvert-\rangle = \tfrac{1}{\sqrt2}(\lvert0\rangle - \lvert1\rangle)$  ii) $\lvert\circlearrowleft\rangle = \tfrac{1}{\sqrt2}(\lvert0\rangle - i\lvert1\rangle)$  iii) $\tfrac{1}{\sqrt2}(\lvert0\rangle + e^{i\pi/4}\lvert1\rangle)$

Use only `x`, `y`, `z`, `h`, `s`, `sdg`, `t`.

In [ ]:
minus = QuantumCircuit(1)
# your gates here
check(minus, [1 / sqrt(2), -1 / sqrt(2)], "i) |->")

circ = QuantumCircuit(1)
# your gates here
check(circ, [1 / sqrt(2), -1j / sqrt(2)], "ii) |↺>")

eighth = QuantumCircuit(1)
# your gates here
check(eighth, [1 / sqrt(2), np.exp(1j * pi / 4) / sqrt(2)], "iii) phase π/4")

### 3.3 Qiskit's bit order: $q_0$ is on the right

In the bitstring `'001'`, the **rightmost** character is qubit 0, just like the least significant bit of a binary number: $\lvert q_2\, q_1\, q_0\rangle$.

In [ ]:
q = QuantumCircuit(3)
q.x(0)
print(Statevector(q).probabilities_dict())
print(Statevector.from_label("001") == Statevector(q))

**Poll from the slides:** predict the output before running.

In [ ]:
r = QuantumCircuit(2)
r.x(1)
print(Statevector(r).probabilities_dict())

### 3.4 `Operator`: Qiskit checks your algebra

`Operator(circuit)` multiplies all the gate matrices into the circuit's unitary. `.equiv()` compares up to a global phase.

In [ ]:
from qiskit.circuit.library import XGate

print(Operator(qc).data.round(3))     # qc is H, Z, H from Part 2
print(Operator(qc).equiv(XGate()))

Check $\mathrm{CX} = (H \otimes I)\,\mathrm{CZ}\,(H \otimes I)$: Hadamards on the **target** turn CZ into CNOT. The printed matrix is CNOT in Qiskit order.

In [ ]:
lhs = QuantumCircuit(2)
lhs.cx(0, 1)

rhs = QuantumCircuit(2)
rhs.h(1)
rhs.cz(0, 1)
rhs.h(1)

print(Operator(lhs).equiv(Operator(rhs)))
print(Operator(lhs).data.real.astype(int))

### 3.5 Bloch spheres

One qubit in $TH\lvert 0\rangle$ has an arrow on the equator, $45^\circ$ from $+x$.

In [ ]:
ht = QuantumCircuit(1)
ht.h(0)
ht.t(0)
print(Statevector(ht))
plot_bloch_multivector(Statevector(ht))

Each qubit of a GHZ state, on its own, is maximally mixed: its Bloch arrow has **length zero**. The information lives only in the correlations.

In [ ]:
plot_bloch_multivector(Statevector(ghz))

### 3.6 Why we need more than `Statevector`

| Qubits | Amplitudes | Memory (16 bytes each) |
|---|---|---|
| 30 | $1.1\times10^{9}$ | 16 GiB |
| 40 | $1.1\times10^{12}$ | 16 TiB |
| 50 | $1.1\times10^{15}$ | 16 PiB |

Real hardware never shows amplitudes, only measurement samples. That's what the **primitives** in Part 4 are for.

### ✏️ Try it (Part 3)

1. Use `Operator(...).equiv(...)` to check $HXH = Z$.
2. Check that $S\,S = Z$ (hint: `from qiskit.circuit.library import ZGate`).
3. Predict, then check: in a 3-qubit circuit with only `x(2)`, which bitstring has probability 1?

In [ ]:
# your code here

---
## Part 4 · Execute: primitives

* **Sampler**: which bitstrings come out, and how often? Input PUB: `(circuit, parameter values)`, plus `shots`.
* **Estimator**: what is the average value $\langle\psi\rvert O\lvert\psi\rangle$? Input PUB: `(circuit, observables, parameter values)`.

A **PUB** (primitive unified bloc) is one unit of work. `run()` takes a **list** of PUBs.

### 4.1 Sampler, line by line

In [ ]:
ghz_m = ghz.copy()        # keep the unmeasured ghz for Statevector / Estimator
ghz_m.measure_all()       # Sampler needs measurements

sampler = StatevectorSampler(seed=7)       # drop seed=7 to get new random results
job = sampler.run([ghz_m], shots=1000)     # returns a job right away
result = job.result()                      # waits for it to finish
counts = result[0].data.meas.get_counts()  # [0] = first PUB, .meas = register name
print(counts)

### 4.2 Unpacking the result

`result` → `[0]` (PubResult) → `.data` (DataBin, one field per classical register) → `.meas` (BitArray) → `.get_counts()` (dict).

In [ ]:
bits = result[0].data.meas
print(bits.num_shots, bits.num_bits)
print(bits.get_bitstrings()[:5])   # shot-by-shot record
print(result[0].data)              # not sure of the register name? look here

### 4.3 Histograms and shot noise

Each shot is a fair coin between `000` and `111`, so the count is binomial with

$$
\sigma = \sqrt{N p (1-p)} = \sqrt{1000 \cdot \tfrac12 \cdot \tfrac12} \approx 15.8 .
$$

In [ ]:
plot_histogram(counts)

### 4.4 Sweep a parameter in one job

$R_y(\theta)\lvert0\rangle = \cos\frac\theta2\lvert0\rangle + \sin\frac\theta2\lvert1\rangle$, so $P(1) = \sin^2\frac\theta2$. One PUB with 21 parameter values runs 21 circuits.

In [ ]:
import matplotlib.pyplot as plt

thetas = np.linspace(0, 2 * np.pi, 21)
res = sampler.run([(dial, thetas)], shots=1000).result()[0]
p1 = [res.data.meas.get_counts(i).get("1", 0) / 1000 for i in range(len(thetas))]

xs = np.linspace(0, 2 * np.pi, 400)
plt.plot(xs, np.sin(xs / 2) ** 2, label="theory sin²(θ/2)")
plt.plot(thetas, p1, "o", label="Sampler, 1000 shots")
plt.xlabel("θ"); plt.ylabel("P(1)"); plt.legend(); plt.show()

### 4.5 Estimator: expectation values directly

Pauli strings use Qiskit order too: `"ZII"` is $Z$ on $q_2$. The Estimator takes the **unmeasured** circuit.

In [ ]:
obs = [SparsePauliOp(p) for p in ["ZZI", "IZZ", "ZII", "XXX"]]
estimator = StatevectorEstimator()
res = estimator.run([(ghz, obs)]).result()[0]
print(res.data.evs)

* $\langle Z_2 Z_1\rangle = \langle Z_1 Z_0\rangle = 1$: neighbouring qubits always agree
* $\langle Z_2\rangle = 0$: each qubit alone is a fair coin
* $\langle XXX\rangle = 1$: $XXX$ maps $\lvert000\rangle \leftrightarrow \lvert111\rangle$, so it leaves the GHZ state unchanged

### 4.6 One job, many circuits: phase is invisible to measurement

`run()` takes a **list** of PUBs, so one job can carry many circuits. Here are three states with the same $p = \tfrac12$ but different phases. Measured in the computational basis, they look **identical**. Telling them apart takes extra gates before the measurement, which is exactly how interference-based algorithms work (Day 2).

In [ ]:
plus_m  = QuantumCircuit(1); plus_m.h(0)
minus_m = QuantumCircuit(1); minus_m.x(0); minus_m.h(0)
circ_m  = QuantumCircuit(1); circ_m.h(0); circ_m.sdg(0)
batch = [c.measure_all(inplace=False) for c in (plus_m, minus_m, circ_m)]

res = StatevectorSampler().run(batch, shots=1000).result()
batch_counts = [res[k].data.meas.get_counts() for k in range(len(batch))]
print(batch_counts)
plot_distribution(batch_counts, legend=["|+>", "|->", "|↺>"])

### ✏️ Try it (Part 4)

1. Sample the GHZ circuit with `shots=4000`. What is $\sigma$ now, and are your counts within about $2\sigma$ of 2000?
2. Use the Estimator to compute $\langle Z_0\rangle$ and $\langle Z_1 Z_0\rangle$ for the state $R_y(\pi/2)\lvert 0\rangle$ on qubit 0 and $\lvert 0\rangle$ on qubit 1. Explain both numbers.

In [ ]:
# your code here

---
## Part 5 · Optimize: transpiling and noise

### 5.1 Rewrite for the chip's own gates

`FakeTorino` is a local snapshot of a real 133-qubit IBM Heron processor: its gate set, connectivity and error rates. It only runs `rz`, `sx`, `x` and `cz`, so the transpiler rewrites our circuit into those (an **ISA** circuit).

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit.transpiler import generate_preset_pass_manager

backend = FakeTorino()
print(backend.num_qubits, backend.operation_names)

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa = pm.run(ghz_m)

print(ghz_m.depth(), dict(ghz_m.count_ops()))
print(isa.depth(), dict(isa.count_ops()))

In [ ]:
isa.draw("mpl", idle_wires=False, fold=-1)

### 5.2 Same code, noisy chip: local testing mode

Passing a fake backend as `mode` runs the job on your laptop with that chip's noise. On **Day 2** you swap `FakeTorino()` for a real IBM backend, and nothing else changes.

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

noisy_sampler = Sampler(mode=backend)
noisy = noisy_sampler.run([isa], shots=1000).result()[0].data.meas.get_counts()
print(dict(sorted(noisy.items(), key=lambda kv: -kv[1])))

plot_histogram([counts, noisy], legend=["ideal (StatevectorSampler)", "noisy (FakeTorino)"])

How much of the result is still "right"? Ideally a GHZ state only ever gives all-equal bitstrings; everything else is **noise**: imperfect gates, readout errors and decoherence.

In [ ]:
def share(counts, keys):
    return sum(counts.get(k, 0) for k in keys) / sum(counts.values())

print(f"ideal: {share(counts, ['000', '111']):.3f}    FakeTorino: {share(noisy, ['000', '111']):.3f}")

### 5.3 The whole workflow in one cell

Map → optimize → execute → post-process. This is the template for every hardware notebook on Day 2.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeTorino

qc = QuantumCircuit(3)                                            # 1. map
qc.h(0); qc.cx(0, 1); qc.cx(1, 2)
qc.measure_all()

backend = FakeTorino()                                            # 2. optimize
isa = generate_preset_pass_manager(optimization_level=1, backend=backend).run(qc)

result = Sampler(mode=backend).run([isa], shots=1000).result()    # 3. execute

counts_hw = result[0].data.meas.get_counts()                      # 4. post-process
print(max(counts_hw, key=counts_hw.get), counts_hw)

### ✏️ Try it (Part 5)

1. Transpile `ghz_m` with `optimization_level=0` and `optimization_level=3`. Compare `depth()` and `count_ops()`.
2. Run the noisy sampler on a **5-qubit** GHZ circuit. What fraction of shots are neither `00000` nor `11111`? Why is it larger than for 3 qubits?

In [ ]:
# your code here

---
## Further reading

* IBM Quantum Documentation: [Development workflow (Qiskit patterns)](https://quantum.cloud.ibm.com/docs/en/guides/intro-to-patterns), [Introduction to primitives](https://quantum.cloud.ibm.com/docs/en/guides/primitives), [Introduction to transpilation](https://quantum.cloud.ibm.com/docs/en/guides/transpile), [Local mode](https://quantum.cloud.ibm.com/docs/en/guides/local-testing-mode), [Bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering)
* Migration guides: [Qiskit 1.0](https://quantum.cloud.ibm.com/docs/en/migration-guides/qiskit-1.0), [Qiskit 2.0](https://quantum.cloud.ibm.com/docs/en/migration-guides/qiskit-2.0)
* IBM Quantum Learning: [Basics of Quantum Information](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information) by John Watrous (single and multiple systems)
* A. Javadi-Abhari *et al.*, "Quantum computing with Qiskit," 2024, [arXiv:2405.08810](https://arxiv.org/abs/2405.08810)

### Coming on Day 2

You'll create an IBM Quantum Platform account (API key and instance CRN), replace `FakeTorino()` with a real backend from `QiskitRuntimeService().least_busy(...)`, save your `job_id` to fetch results later, and try error suppression and mitigation options on the Sampler.

---
## Solutions to the Try it exercises

Try the exercises first!

### Part 2

In [ ]:
# 1. 4-qubit GHZ: depth 4 (H, then 3 CNOTs that must wait for each other), size 4
g4 = QuantumCircuit(4)
g4.h(0)
for i in range(3):
    g4.cx(i, i + 1)
print("GHZ-4 depth:", g4.depth(), " size:", g4.size())

# 2. Three Hadamards act on different qubits, so they all fit in ONE layer
h3 = QuantumCircuit(3)
h3.h([0, 1, 2])
print("H on every qubit, depth:", h3.depth(), " size:", h3.size())

### Part 3

In [ ]:
from qiskit.circuit.library import ZGate

hxh = QuantumCircuit(1); hxh.h(0); hxh.x(0); hxh.h(0)
print("HXH = Z ?", Operator(hxh).equiv(ZGate()))

ss = QuantumCircuit(1); ss.s(0); ss.s(0)
print("SS = Z ?", Operator(ss).equiv(ZGate()))

x2 = QuantumCircuit(3); x2.x(2)
print(Statevector(x2).probabilities_dict())   # '100': qubit 2 is the LEFTMOST character

### Part 3 practice (qsphere states)

i) $X$ then $H$ (or $H$ then $Z$). ii) $H$ then $S^\dagger$. iii) $H$ then $T$, because $T$ adds $\pi/4$ to the phase.

In [ ]:
minus = QuantumCircuit(1); minus.x(0); minus.h(0)
check(minus, [1 / sqrt(2), -1 / sqrt(2)], "i) |->")
circ = QuantumCircuit(1); circ.h(0); circ.sdg(0)
check(circ, [1 / sqrt(2), -1j / sqrt(2)], "ii) |↺>")
eighth = QuantumCircuit(1); eighth.h(0); eighth.t(0)
check(eighth, [1 / sqrt(2), np.exp(1j * pi / 4) / sqrt(2)], "iii) phase π/4")

### Part 4

With 4000 shots, $\sigma = \sqrt{4000 \cdot \frac12 \cdot \frac12} \approx 31.6$.

For $R_y(\pi/2)\lvert0\rangle$ on $q_0$: $\langle Z_0\rangle = \cos^2\frac{\pi}{4} - \sin^2\frac{\pi}{4} = 0$. Qubit 1 is in $\lvert 0\rangle$, so $\langle Z_1\rangle = 1$ and the product state gives $\langle Z_1 Z_0\rangle = \langle Z_1\rangle\langle Z_0\rangle = 0$.

In [ ]:
c4000 = StatevectorSampler().run([ghz_m], shots=4000).result()[0].data.meas.get_counts()
sigma = (4000 * 0.5 * 0.5) ** 0.5
print(c4000, " sigma =", round(sigma, 1), " within 2 sigma:", abs(c4000.get("000", 0) - 2000) <= 2 * sigma)

ry = QuantumCircuit(2)
ry.ry(pi / 2, 0)
evs = StatevectorEstimator().run([(ry, [SparsePauliOp("IZ"), SparsePauliOp("ZZ")])]).result()[0].data.evs
print("<Z0> =", evs[0].round(6), "  <Z1 Z0> =", evs[1].round(6))

### Part 5

More qubits means more gates (each one a chance for an error) and more qubits to misread, so the fraction of wrong bitstrings grows with $n$.

In [ ]:
backend = FakeTorino()
for level in (0, 3):
    t = generate_preset_pass_manager(optimization_level=level, backend=backend).run(ghz_m)
    print(f"optimization_level={level}: depth={t.depth()}, ops={dict(t.count_ops())}")

g5 = QuantumCircuit(5)
g5.h(0)
for i in range(4):
    g5.cx(i, i + 1)
g5.measure_all()
isa5 = generate_preset_pass_manager(optimization_level=1, backend=backend).run(g5)
c5 = Sampler(mode=backend).run([isa5], shots=1000).result()[0].data.meas.get_counts()
wrong = 1 - (c5.get("00000", 0) + c5.get("11111", 0)) / 1000
print(f"5-qubit GHZ: {wrong:.0%} of shots are neither 00000 nor 11111")

---
## Credits

Prepared for **Qiskit Fall Fest 2026 at UT San Antonio** by Van Tien Nguyen, to accompany the session *Getting Started with Qiskit*.